# PUF256 同 bitmap 雙板驗證 — 20100

使用原始已通過時序檢查的 bitstream。共同 bitmap 僅依兩板訓練資料的穩定度與 margin 選擇；凍結後才做新的 holdout、硬體註冊與重建。每板使用自身 helper。

載入 overlay 會取代目前 PL 設計；本 Notebook 只對指定借用板操作。

In [1]:
from pathlib import Path
import json, hashlib
root = Path.cwd()
for manifest_name in ['MANIFEST.json', 'CROSS_MANIFEST.json']:
    manifest = json.loads((root / manifest_name).read_text())
    for name, record in manifest['files'].items():
        if name.endswith('.ipynb'): continue
        data = (root / name).read_bytes()
        assert len(data) == record['bytes']
        assert hashlib.sha256(data).hexdigest() == record['sha256'], name
print('Base overlay and frozen cross-board inputs verified')


Base overlay and frozen cross-board inputs verified


In [2]:
from cross_board import run_fixed
summary = run_fixed('20100', holdout_reads=32, reconstruction_runs=100, reload_runs=10)
assert summary['passed']


20100 fixed-bitmap holdout 8 / 32
20100 fixed-bitmap holdout 16 / 32
20100 fixed-bitmap holdout 24 / 32
20100 fixed-bitmap holdout 32 / 32
20100 common-bitmap reconstructions 10 / 100
20100 common-bitmap reconstructions 20 / 100
20100 common-bitmap reconstructions 30 / 100
20100 common-bitmap reconstructions 40 / 100
20100 common-bitmap reconstructions 50 / 100
20100 common-bitmap reconstructions 60 / 100
20100 common-bitmap reconstructions 70 / 100
20100 common-bitmap reconstructions 80 / 100
20100 common-bitmap reconstructions 90 / 100
20100 common-bitmap reconstructions 100 / 100
20100 common-bitmap reload 1 / 10
20100 common-bitmap reload 2 / 10
20100 common-bitmap reload 3 / 10
20100 common-bitmap reload 4 / 10
20100 common-bitmap reload 5 / 10
20100 common-bitmap reload 6 / 10
20100 common-bitmap reload 7 / 10
20100 common-bitmap reload 8 / 10
20100 common-bitmap reload 9 / 10
20100 common-bitmap reload 10 / 10
{
  "passed": true,
  "board_label": "20100",
  "bitmap_hex": "feebfe

## 匯出證據

身份碼來自實際 PL 暫存器，CPU 只做輸入、量測記錄與實驗比對。結果不是跨晶片母體唯一性或熵的估計。

In [3]:
import zipfile
from IPython.display import display, FileLink
latest = sorted(p for p in root.glob('common_20100_*') if p.is_dir())[-1]
print((latest / 'summary.json').read_text())
archive = root / (latest.name + '.zip')
with zipfile.ZipFile(str(archive), 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(latest.iterdir()):
        if path.is_file(): bundle.write(str(path), latest.name + '/' + path.name)
display(FileLink(archive.name))


{
  "bitmap_hex": "feebfe3ffff7fbedfdfbe7fbbfbbeced7ffaf1fbbc37836dfbffbb7ff4ffeafd97bfffbdfbefdfff",
  "bitstream_sha256": "79346d7855c4a678563e8010083456b762b63bc8a8455388b2c45d4f12b7c7c2",
  "board_label": "20100",
  "candidate_hex": "0c5650309cdaafb478690aea91892220e67fb24cb3a18685df719a04d007861d",
  "corrected_words": 0,
  "elapsed_seconds": 172.56695067230612,
  "holdout_samples": 8192,
  "key_sha256": "2e8cabe88c015c46363a8e566d758b2ec14a614605758c33b052f79ebc1570c2",
  "passed": true,
  "plan_sha256": "a3530b72719b7b8c983a95f7a1afca6f3c71c2d46d406b32987236ffadb57346",
  "reconstruction_count": 111,
  "reload_count": 10
}


/home/xilinx/jupyter_notebooks/puf256_compare_a0981_20261113/common_20100_20261112_220713.zip